In [24]:
import numpy as np
import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
import torchvision.transforms as T

## Network Architecture

In [25]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.convnet = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 320x320

            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 160x160

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 80x80

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 40x40

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(512, 128)

    def forward(self, x):
        x = self.convnet(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)  # batch_size, 128)
        return x
    
transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
])

## import model weights

In [26]:
embedding_net = EmbeddingNet()
embedding_net.load_state_dict(torch.load("embedding_net.pth", map_location="cpu"))
embedding_net.eval()

EmbeddingNet(
  (convnet): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (fc): Linear(in_features=512, out_features=128, bias=True)
)

## Define Metric Functions

In [27]:
def compute_distances(anchor_embed, img_dir, model, transform, device):
    distances, files = [], []
    for fname in os.listdir(img_dir):
        fpath = os.path.join(img_dir, fname)
        if not os.path.isfile(fpath):
            continue
        img = Image.open(fpath).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embed = model(img_tensor)
            dist = F.pairwise_distance(anchor_embed, embed).item()
            distances.append(dist)
            files.append(fname)
    return distances, files


def summarize(distances, files):
    if not distances:
        return None
    distances = np.array(distances)
    min_idx = np.argmin(distances)
    max_idx = np.argmax(distances)
    return {
        "min": float(distances[min_idx]),
        "min_file": files[min_idx],
        "max": float(distances[max_idx]),
        "max_file": files[max_idx],
        "mean": float(np.mean(distances)),
    }


def evaluate_anchor(anchor_path, positive_dir, negative_dir, model, transform, device="cpu"):
    model = model.to(device)
    model.eval()
    anchor_img = Image.open(anchor_path).convert("RGB")
    anchor_tensor = transform(anchor_img).unsqueeze(0).to(device)
    with torch.no_grad():
        anchor_embed = model(anchor_tensor)
    pos_distances, pos_files = compute_distances(anchor_embed, positive_dir, model, transform, device)
    neg_distances, neg_files = compute_distances(anchor_embed, negative_dir, model, transform, device)
    return {
        "positive": summarize(pos_distances, pos_files),
        "negative": summarize(neg_distances, neg_files),
    }


## Test

In [29]:
NUMBER_TEST = 15

for i in range(1, NUMBER_TEST):
    anchor_path = f"triplet_data_test/Anchor/{i}.jpg"
    positive_dir = "triplet_data_test/Positive"
    negative_dir = "triplet_data_test/Negative"

    results = evaluate_anchor(anchor_path, positive_dir, negative_dir, embedding_net, transform)
    positive_distance = results["positive"]
    negative_distance = results['negative']
    answer = f"Test id: {i} -> Positive distances: {positive_distance} - Negative distance: {negative_distance}\n"
    print(answer)

Test id: 1 -> Positive distances: {'min': 1.131371027440764e-05, 'min_file': '2.jpg', 'max': 5.2955522537231445, 'max_file': '13.jpg', 'mean': 2.297919997821494} - Negative distance: {'min': 3.5124258995056152, 'min_file': '10.jpg', 'max': 14.731945037841797, 'max_file': '7.jpg', 'mean': 7.385353660583496}

Test id: 2 -> Positive distances: {'min': 0.020539497956633568, 'min_file': '9.jpg', 'max': 3.755972146987915, 'max_file': '15.jpg', 'mean': 1.7608346128215393} - Negative distance: {'min': 0.6654590368270874, 'min_file': '10.jpg', 'max': 11.879457473754883, 'max_file': '7.jpg', 'mean': 4.5332605282465614}

Test id: 3 -> Positive distances: {'min': 1.131371027440764e-05, 'min_file': '6.jpg', 'max': 4.532127857208252, 'max_file': '15.jpg', 'mean': 1.9436376524470689} - Negative distance: {'min': 0.1515364646911621, 'min_file': '10.jpg', 'max': 11.103299140930176, 'max_file': '7.jpg', 'mean': 3.7746172189712524}

Test id: 4 -> Positive distances: {'min': 1.131371027440764e-05, 'min_fi